# Agente Text-to-SQL — CineData Analytics

Este notebook implementa um agente Text-to-SQL utilizando PydanticAI, OpenRouter e SQLite.

O fluxo foi inspirado de forma simplificada na arquitetura CHESS, separando etapas de schema linking, geração de SQL, execução, validação e correção de consultas.

## 1. Configuração do ambiente

Nesta seção são carregadas as dependências, variáveis de ambiente e configurações gerais utilizadas pelo agente.

In [114]:
import os
import re
import asyncio
import sqlite3
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai import Agent, PromptedOutput
from pydantic_ai.exceptions import UnexpectedModelBehavior


load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY não encontrada no arquivo .env")

In [141]:
# Carregando constantes do projeto
MODEL_NAME = "nvidia/nemotron-3.5-lightning:free"
DATABASE_PATH = Path("cinerocket.db")
DB_PATH = str(DATABASE_PATH)

In [71]:
# Validando se o banco está disponível localmente
if not DATABASE_PATH.exists():
    raise FileNotFoundError(
        f"Banco de dados não encontrado em: {DATABASE_PATH.resolve()}"
    )


### 1.2 Configuração do modelo

O modelo é acessado através da API da OpenRouter, utilizando uma interface compatível com a API da OpenAI.

In [72]:
pydantic_model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    ),
)

In [73]:
# Testando modelo
test_agent = Agent(
    pydantic_model,
    instructions="Responda apenas com OK",
)

result = await test_agent.run("Teste de conexão")
print(result.output)

OK


## 2. Utilitários do banco de dados

Funções auxiliares responsáveis pela conexão e inspeção do banco SQLite.

Essas funções são mantidas separadas da lógica dos agentes para facilitar testes, reutilização e manutenção.

In [74]:
# Funçao de conexão com banco
def get_connection(database_path: Path = DATABASE_PATH) -> sqlite3.Connection:
    """
    Cria uma conexão com o banco SQLite utilizado pelo agente.

    Args:
        database_path: Caminho para o arquivo SQLite.

    Returns:
        Uma conexão ativa com o banco de dados SQLite.

    Raises:
        FileNotFoundError: Caso o arquivo do banco não exista.
    """
    if not database_path.exists():
        raise FileNotFoundError(
            f"Banco de dados não encontrado: {database_path}"
        )

    return sqlite3.connect(database_path)

In [75]:
# Pegando os nomes das tabelas do banco
def get_table_names(conn: sqlite3.Connection) -> list[str]:
    """
    Retorna os nomes das tabelas disponíveis no banco SQLite.

    Args:
        conn: Conexão ativa com o banco SQLite.

    Returns:
        Lista contendo os nomes das tabelas encontradas.
    """
    query = """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name;
    """

    cursor = conn.execute(query)

    return [row[0] for row in cursor.fetchall()]

In [76]:
# Verificando se as tabelas foram carregadas corretamente
with get_connection() as conn:
    tables = get_table_names(conn)

tables

['alembic_version',
 'bridge_movie_company',
 'bridge_movie_genre',
 'bridge_movie_person',
 'dim_companies',
 'dim_genres',
 'dim_movies',
 'dim_people',
 'dim_reviews',
 'fact_movies_performance',
 'movie_reviews']

### 2.1 Inspeção do esquema das tabelas

In [77]:
def get_table_schema(
    conn: sqlite3.Connection,
    table_name: str,
) -> list[dict[str, Any]]:
    """
    Retorna informações sobre as colunas de uma tabela SQLite.

    Args:
        conn: Conexão ativa com o banco SQLite.
        table_name: Nome da tabela que será inspecionada.

    Returns:
        Lista de dicionários contendo informações sobre as colunas.

    Raises:
        ValueError: Caso a tabela informada não exista no banco.
    """
    available_tables = get_table_names(conn)

    if table_name not in available_tables:
        raise ValueError(
            f"Tabela '{table_name}' não encontrada no banco."
        )

    cursor = conn.execute(f"PRAGMA table_info({table_name});")

    columns = []

    for row in cursor.fetchall():
        columns.append(
            {
                "name": row[1],
                "type": row[2],
                "not_null": bool(row[3]),
                "primary_key": bool(row[5]),
            }
        )

    return columns


def get_database_schema(
    db_path: str,
    ignored_tables: set[str] | None = None,
) -> str:
    """
    Retorna uma representação textual compacta do esquema do banco SQLite.

    A função lista tabelas e respectivas colunas para uso no processo
    de schema linking. Tabelas técnicas podem ser ignoradas.

    Args:
        db_path: Caminho para o arquivo SQLite.
        ignored_tables: Conjunto opcional de tabelas que não devem
            aparecer no esquema.

    Returns:
        Representação textual do esquema do banco.

    Raises:
        FileNotFoundError: Caso o arquivo do banco não exista.
    """
    if ignored_tables is None:
        ignored_tables = {"alembic_version"}

    if not Path(db_path).exists():
        raise FileNotFoundError(f"Banco não encontrado: {db_path}")

    conn = sqlite3.connect(db_path)

    try:
        cursor = conn.cursor()

        cursor.execute(
            """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name;
            """
        )

        tables = [
            row[0]
            for row in cursor.fetchall()
            if row[0] not in ignored_tables
        ]

        schema_parts: list[str] = []

        for table in tables:
            cursor.execute(f'PRAGMA table_info("{table}")')

            columns = cursor.fetchall()

            schema_parts.append(f"Tabela: {table}")

            for column in columns:
                name = column[1]
                data_type = column[2]
                not_null = bool(column[3])
                primary_key = bool(column[5])

                metadata: list[str] = []

                if primary_key:
                    metadata.append("PK")

                if not_null:
                    metadata.append("NOT NULL")

                suffix = (
                    f" [{' | '.join(metadata)}]"
                    if metadata
                    else ""
                )

                schema_parts.append(
                    f"  - {name}: {data_type}{suffix}"
                )

            schema_parts.append("")

        return "\n".join(schema_parts)

    finally:
        conn.close()

> **Observação:** o esquema abaixo é utilizado apenas na etapa de *schema linking*.  
> O gerador de SQL receberá posteriormente apenas o subconjunto do esquema considerado relevante para cada pergunta.

In [78]:
schema = get_database_schema(DB_PATH)
print(schema)

Tabela: bridge_movie_company
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - sk_company_id: VARCHAR(64) [PK | NOT NULL]

Tabela: bridge_movie_genre
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - sk_genre_id: VARCHAR(64) [PK | NOT NULL]

Tabela: bridge_movie_person
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - sk_person_id: VARCHAR(64) [PK | NOT NULL]

Tabela: dim_companies
  - sk_company_id: VARCHAR(64) [PK | NOT NULL]
  - nome_produtora: VARCHAR(255) [NOT NULL]

Tabela: dim_genres
  - sk_genre_id: VARCHAR(64) [PK | NOT NULL]
  - nome_genero: VARCHAR(50) [NOT NULL]

Tabela: dim_movies
  - sk_movie_id: VARCHAR(64) [PK | NOT NULL]
  - id_filme: VARCHAR(50) [NOT NULL]
  - titulo: VARCHAR(500) [NOT NULL]
  - data_lancamento: DATE
  - ano_lancamento: INTEGER
  - duracao_minutos: INTEGER
  - idioma_original: VARCHAR(10)
  - status_filme: VARCHAR(50)
  - sinopse: VARCHAR(4000)
  - url_poster: VARCHAR(2048)
  - url_backdrop: VARCHAR(2048)

Tabela: dim_people
  - sk_person_id: VARCHAR(64)

## 3. Representação dos relacionamentos

Como o banco utiliza tabelas dimensão, fato e tabelas de associação, os principais relacionamentos são explicitados para auxiliar o processo de schema linking.

In [79]:
# Os relacionamentos são utilizados como contexto estrutural e não representam dados produzidos pelo modelo
RELATIONSHIPS = [
    "bridge_movie_company.sk_movie_id = dim_movies.sk_movie_id",
    "bridge_movie_company.sk_company_id = dim_companies.sk_company_id",

    "bridge_movie_genre.sk_movie_id = dim_movies.sk_movie_id",
    "bridge_movie_genre.sk_genre_id = dim_genres.sk_genre_id",

    "bridge_movie_person.sk_movie_id = dim_movies.sk_movie_id",
    "bridge_movie_person.sk_person_id = dim_people.sk_person_id",

    "fact_movies_performance.sk_movie_id = dim_movies.sk_movie_id",

    "dim_reviews.sk_movie_id = dim_movies.sk_movie_id",

    "movie_reviews.sk_movie_id = dim_movies.sk_movie_id",
]

## 4. Schema Linking

O schema linking identifica o conjunto mínimo de tabelas, colunas, filtros e relacionamentos necessários para responder a uma pergunta em linguagem natural.

Essa etapa reduz o contexto fornecido posteriormente ao gerador de SQL.

In [80]:
# Modelo de saída do schema linking
class SchemaLink(BaseModel):
    """Elementos mínimos do esquema necessários para responder à pergunta."""

    tables: list[str] = Field(
        description="Nomes das tabelas necessárias para responder à pergunta."
    )

    columns: list[str] = Field(
        description=(
            "Colunas necessárias no formato 'tabela.coluna'. "
            "Exemplo: 'dim_movies.titulo'."
        )
    )

    filters: list[str] = Field(
        default_factory=list,
        description=(
            "Condições de filtro necessárias para responder à pergunta. "
            "Não invente valores que não estejam presentes na pergunta."
        )
    )

    joins: list[str] = Field(
    default_factory=list,
    description=(
        "Relacionamentos necessários entre as tabelas. "
        "Use referências 'tabela.coluna' quando houver tabelas distintas. "
        "Quando a mesma tabela precisar participar mais de uma vez, "
        "descreva semanticamente a auto-junção sem criar aliases SQL."
        )
    )

    reasoning: str = Field(
        description=(
            "Explicação breve, em português Br, de por que os elementos foram selecionados."
        )
    )
    

# Constrói o contexto global do schema linker
def get_schema_for_linking(db_path: str) -> str:
    """
    Constrói o contexto global utilizado pelo agente de schema linking.

    Inclui a estrutura das tabelas analíticas e os relacionamentos
    conhecidos entre elas.

    Args:
        db_path: Caminho para o banco SQLite.

    Returns:
        Contexto textual contendo esquema e relacionamentos.
    """
    schema = get_database_schema(db_path)

    relationships_text = "\n".join(
        f"- {relationship}"
        for relationship in RELATIONSHIPS
    )

    return f"""
ESQUEMA:

{schema}

RELACIONAMENTOS:

{relationships_text}
""".strip()

### 4.1 Agente de schema linking

In [81]:
schema_link_agent = Agent(
    pydantic_model,
    output_type=PromptedOutput(SchemaLink),
    instructions=f"""
Você é responsável pelo schema linking de um sistema Text-to-SQL.

Sua única tarefa é identificar o MENOR conjunto de elementos do banco
necessário para responder à pergunta do usuário.

Use exclusivamente o esquema fornecido.

Regras:

1. Nunca invente tabelas ou colunas.
2. Retorne apenas tabelas existentes no esquema.
3. Toda coluna deve estar no formato "tabela.coluna".
4. Se uma pergunta exigir mais de uma tabela, identifique os joins necessários.
5. Em filters, descreva apenas filtros necessários para responder à pergunta.
6. Não invente valores de filtro que não estejam explicitamente presentes na pergunta.
7. Não gere SQL.
8. Não responda à pergunta do usuário.
9. Se uma tabela bridge for necessária para conectar entidades, inclua-a.
10. Utilize apenas os relacionamentos fornecidos.
11. Prefira sempre o menor conjunto possível de tabelas e colunas.
12. Preencha todos os campos da saída estruturada.

13. Não inclua tabelas intermediárias que não sejam necessárias para conectar
    os elementos selecionados.

14. Quando um filtro depender de valores categóricos do banco que não estejam
    explicitamente escritos na pergunta, NÃO escreva valores literais.

    Exemplo incorreto:
    "dim_people.tipo_pessoa = 'ator'"

    Exemplo correto:
    "filtrar dim_people.tipo_pessoa para representar pessoas com papel de ator"

    Os valores reais serão consultados diretamente no banco em outra etapa.

15. Uma mesma tabela pode ser necessária em múltiplos papéis lógicos.
    Nesse caso, inclua a tabela apenas uma vez em tables.

16. Não inclua uma tabela apenas por integridade referencial ou contexto conceitual.
    Inclua somente tabelas necessárias para obter colunas, aplicar filtros ou
    estabelecer os relacionamentos necessários à resposta.

    Se o resultado puder ser obtido sem uma tabela, ela deve ser omitida.

17. Não crie aliases SQL no schema linking. Descreva apenas as relações
    estruturais necessárias. A criação de aliases será responsabilidade
    do gerador de SQL.

18. Em reasoning, explique apenas por que os elementos do esquema foram
    selecionados. Não descreva a implementação completa da consulta SQL.

{get_schema_for_linking(DB_PATH)}
""",
)

### 4.2 Teste do agente de schema linking

In [82]:
question = "Top 10 filmes com maior receita em R$"

result = await schema_link_agent.run(question)

linked = result.output

print("Tables:   ", linked.tables)
print("Columns:  ", linked.columns)
print("Filters:  ", linked.filters)
print("Joins:    ", linked.joins)
print("Reasoning:", linked.reasoning)

Tables:    ['dim_movies', 'fact_movies_performance']
Columns:   ['dim_movies.titulo', 'fact_movies_performance.receita_brl', 'dim_movies.sk_movie_id', 'fact_movies_performance.sk_movie_id']
Filters:   []
Joins:     ['dim_movies.sk_movie_id = fact_movies_performance.sk_movie_id']
Reasoning: A pergunta pede os top 10 filmes por receita em reais. O título do filme está em dim_movies.titulo e a receita em BRL está em fact_movies_performance.receita_brl. As duas tabelas se conectam via sk_movie_id. Não há filtros adicionais; apenas ordenação decrescente por receita_brl e limite de 10 registros.


In [83]:
question = "Qual dupla ator-diretor mais trabalhou junta?"

result = await schema_link_agent.run(question)
linked = result.output

print("Tables:   ", linked.tables)
print("Columns:  ", linked.columns)
print("Filters:  ", linked.filters)
print("Joins:    ", linked.joins)
print("Reasoning:", linked.reasoning)

Tables:    ['dim_people', 'bridge_movie_person']
Columns:   ['dim_people.nome_pessoa', 'dim_people.tipo_pessoa', 'dim_people.sk_person_id', 'bridge_movie_person.sk_movie_id', 'bridge_movie_person.sk_person_id']
Filters:   ['filtrar dim_people.tipo_pessoa para representar pessoas com papel de ator', 'filtrar dim_people.tipo_pessoa para representar pessoas com papel de diretor']
Joins:     ['bridge_movie_person.sk_person_id = dim_people.sk_person_id (para atores)', 'bridge_movie_person.sk_person_id = dim_people.sk_person_id (para diretores)', 'bridge_movie_person (ator).sk_movie_id = bridge_movie_person (diretor).sk_movie_id']
Reasoning: Para encontrar a dupla ator-diretor que mais trabalhou junta, preciso identificar atores e diretores na dim_people, vincular cada um aos filmes via bridge_movie_person, encontrar filmes em comum entre cada par ator-diretor e contar quantos filmes compartilham. A dim_people é usada duas vezes (ator e diretor) e a bridge_movie_person também duas vezes, uni

## 5. Construção do focused_context

Após o schema linking, apenas as tabelas e colunas selecionadas são utilizadas para construir o contexto enviado ao gerador de SQL.

Isso reduz informações irrelevantes no prompt e ajuda a diminuir a geração de tabelas ou colunas inexistentes.

In [84]:
# Seleciona colunas relevantes
def get_columns_for_table(
    linked: SchemaLink,
    table: str,
) -> list[str]:
    """
    Retorna as colunas relevantes de uma tabela.

    Inclui tanto as colunas selecionadas explicitamente pelo schema linker
    quanto as colunas necessárias para realizar os joins.

    Args:
        linked: Resultado estruturado do schema linking.
        table: Nome da tabela.

    Returns:
        Lista ordenada com os nomes das colunas relevantes.
    """
    prefix = f"{table}."

    selected_columns = {
        column.removeprefix(prefix)
        for column in linked.columns
        if column.startswith(prefix)
    }

    join_columns = set(
        get_join_columns(linked, table)
    )

    return sorted(
        selected_columns | join_columns
    )


# Auxilia get_columns_for_table extraindo as colunas nos joins
def get_join_columns(
    linked: SchemaLink,
    table: str,
) -> list[str]:
    """
    Extrai as colunas de uma tabela mencionadas nos joins identificados
    pelo schema linker.

    Args:
        linked: Resultado estruturado do schema linking.
        table: Nome da tabela.

    Returns:
        Lista de nomes de colunas da tabela utilizadas em joins.
    """
    pattern = rf"\b{re.escape(table)}\.([A-Za-z_][A-Za-z0-9_]*)\b"

    columns: set[str] = set()

    for join in linked.joins:
        columns.update(re.findall(pattern, join))

    return sorted(columns) 
 
# Excluindo chaves surrogate sem removê-las da definição de colunas
def get_sample_columns(columns: list[str]) -> list[str]:
    """
    Remove das amostras colunas técnicas de chave surrogate.

    As colunas continuam presentes no contexto estrutural, mas seus valores
    não são enviados ao modelo para evitar consumo desnecessário de tokens.

    Args:
        columns: Lista de colunas relevantes.

    Returns:
        Lista de colunas apropriadas para amostragem.
    """
    return [
        column
        for column in columns
        if not column.startswith("sk_")
    ]

   
# Pega uma amostra de linhas   
def get_sample_rows(
    db_path: str,
    table: str,
    columns: list[str],
    limit: int = 3,
) -> list[dict[str, object]]:
    """
    Obtém uma pequena amostra de linhas das colunas relevantes.

    Args:
        db_path: Caminho para o banco SQLite.
        table: Nome da tabela consultada.
        columns: Colunas que devem aparecer na amostra.
        limit: Número máximo de linhas retornadas.

    Returns:
        Lista de dicionários representando as linhas consultadas.
    """
    
    if not columns:
        return []

    quoted_columns = ", ".join(
        f'"{column}"'
        for column in columns
    )

    query = f"""
        SELECT {quoted_columns}
        FROM "{table}"
        LIMIT ?
    """

    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row

    try:
        cursor = conn.execute(query, (limit,))

        return [
            dict(row)
            for row in cursor.fetchall()
        ]

    finally:
        conn.close()
        
# Valores categóricos relevantes
VALUE_SAMPLE_COLUMNS = {
    "dim_people.tipo_pessoa",
    "dim_movies.status_filme",
    "dim_movies.idioma_original",
}

       
def get_column_values(
    db_path: str,
    table: str,
    column: str,
    limit: int = 10,
) -> list[object]:
    """
    Retorna valores distintos não nulos de uma coluna.

    Args:
        db_path: Caminho para o banco SQLite.
        table: Nome da tabela.
        column: Nome da coluna.
        limit: Quantidade máxima de valores retornados.

    Returns:
        Lista de valores distintos encontrados na coluna.
    """
    
    if limit <= 0:
        return []
    
    query = f"""
        SELECT DISTINCT "{column}"
        FROM "{table}"
        WHERE "{column}" IS NOT NULL
        LIMIT ?
    """

    conn = sqlite3.connect(db_path)

    try:
        cursor = conn.execute(query, (limit,))
        return [row[0] for row in cursor.fetchall()]

    finally:
        conn.close()
        
# Montando o focused_context  
def build_focused_context(
    linked: SchemaLink,
    db_path: str,
) -> str:
    """
    Constrói o contexto mínimo utilizado pelo gerador SQL.

    O contexto inclui:
    - tabelas selecionadas pelo schema linker;
    - colunas relevantes;
    - pequenas amostras dessas colunas;
    - joins necessários;
    - filtros semânticos;
    - valores categóricos observados no banco.

    Args:
        linked: Resultado estruturado do schema linking.
        db_path: Caminho para o banco SQLite.

    Returns:
        Contexto textual focado para geração da consulta SQL.
    """
    parts: list[str] = []

    for table in linked.tables:
        relevant_columns = get_columns_for_table(
            linked,
            table,
        )

        if not relevant_columns:
            continue

        parts.append(
            f"### TABELA: {table}"
        )

        columns_text = "\n".join(
            f"- {column}"
            for column in relevant_columns
        )

        parts.append(
            f"Colunas relevantes:\n{columns_text}"
        )

        samples_columns = get_sample_columns(
            relevant_columns
        )

        samples = get_sample_rows(
            db_path,
            table,
            samples_columns,
        )

        if samples:
            parts.append(
                f"Amostras relevantes: {samples}"
            )

    if linked.joins:
        parts.append(
            "### JOINS NECESSÁRIOS"
        )

        parts.extend(
            f"- {join}"
            for join in linked.joins
        )

    if linked.filters:
        parts.append(
            "### FILTROS IDENTIFICADOS"
        )

        parts.extend(
            f"- {filter_condition}"
            for filter_condition in linked.filters
        )

    values_section: list[str] = []

    for column_ref in linked.columns:
        if column_ref not in VALUE_SAMPLE_COLUMNS:
            continue

        table, column = column_ref.split(".", 1)

        values = get_column_values(
            db_path,
            table,
            column,
        )

        if values:
            values_section.append(
                f"{column_ref}: {values}"
            )

    if values_section:
        parts.append(
            "### VALORES CATEGÓRICOS OBSERVADOS"
        )

        parts.extend(values_section)

    return "\n\n".join(parts)    


### 5.1 Teste do focused_context

In [85]:
question = "Qual dupla ator-diretor mais trabalhou junta?"

result = await schema_link_agent.run(question)
linked = result.output

context = build_focused_context(
    linked,
    DB_PATH,
)

print(context)

### TABELA: dim_movies

Colunas relevantes:
- sk_movie_id

### TABELA: bridge_movie_person

Colunas relevantes:
- sk_movie_id
- sk_person_id

### TABELA: dim_people

Colunas relevantes:
- nome_pessoa
- sk_person_id
- tipo_pessoa

Amostras relevantes: [{'nome_pessoa': '"" The Ripping Movie Of The Summer In Theaters!"', 'tipo_pessoa': 'Roteirista'}, {'nome_pessoa': '"""you Win Some. You Lose Most."""', 'tipo_pessoa': 'Diretor'}, {'nome_pessoa': '""loro"" Wishes To Tell The Story Of Some Italians', 'tipo_pessoa': 'Roteirista'}]

### JOINS NECESSÁRIOS

- dim_movies.sk_movie_id = bridge_movie_person.sk_movie_id

- bridge_movie_person.sk_person_id = dim_people.sk_person_id

### FILTROS IDENTIFICADOS

- filtrar dim_people.tipo_pessoa para representar pessoas com papel de ator

- filtrar dim_people.tipo_pessoa para representar pessoas com papel de diretor

### VALORES CATEGÓRICOS OBSERVADOS

dim_people.tipo_pessoa: ['Roteirista', 'Diretor', 'Ator']


## 6. Geração da consulta SQL

Nesta etapa, um segundo agente recebe apenas o contexto selecionado pelo schema linking e gera uma consulta SQL para responder à pergunta do usuário.

In [86]:
class SQLGeneration(BaseModel):
    """Resultado da etapa de geração da consulta SQL."""

    sql: str = Field(
        description="Consulta SQL de leitura compatível com SQLite."
    )

    reasoning: str = Field(
        description=(
            "Explicação breve, em português Br, da estratégia utilizada para construir a consulta, "
            "sem inventar informações não presentes no contexto."
        )
    )

In [87]:
def build_sql_generation_agent(
    context: str,
) -> Agent:
    """
    Cria um agente responsável por gerar SQL a partir de um contexto focado.

    Args:
        context: Contexto contendo apenas tabelas, colunas, joins,
            filtros e valores relevantes para a pergunta atual.

    Returns:
        Agente configurado para geração estruturada de SQL.
    """
    return Agent(
        pydantic_model,
        output_type=PromptedOutput(SQLGeneration),
        instructions=f"""
Você é responsável pela geração de consultas SQL em um sistema Text-to-SQL.

Sua tarefa é gerar UMA consulta SQL SQLite que responda corretamente
à pergunta do usuário.

Use exclusivamente o contexto fornecido abaixo.

Regras:

1. Gere apenas consultas de leitura.
2. A consulta deve começar com SELECT ou WITH.
3. Nunca utilize INSERT, UPDATE, DELETE, DROP, ALTER, CREATE ou qualquer
   outro comando que modifique o banco.
4. Nunca invente tabelas ou colunas.
5. Utilize somente tabelas e colunas presentes no contexto.
6. Utilize somente valores categóricos explicitamente observados no contexto.
7. Não invente valores existentes no banco.
8. Utilize os joins fornecidos quando necessários.
9. Quando a mesma tabela precisar desempenhar papéis diferentes,
   utilize aliases SQL distintos e faça os joins necessários entre
   essas ocorrências.
10. Utilize sintaxe compatível com SQLite.
11. Quando a pergunta solicitar ranking, utilize ORDER BY e LIMIT
    quando apropriado.
12. Quando a pergunta solicitar média, soma, contagem ou outra agregação,
    utilize a função de agregação correspondente.
13. Trate valores NULL quando isso for necessário para responder corretamente
    à pergunta.
14. Não consulte tabelas ou colunas fora do contexto fornecido.
15. Não responda à pergunta em linguagem natural. Apenas produza a consulta
    e uma explicação breve da estratégia no campo reasoning.
16. Quando duas pessoas precisarem ser relacionadas pelo mesmo filme,
    utilize o identificador do filme disponível na tabela de associação
    para conectar as diferentes ocorrências.

CONTEXTO DISPONÍVEL:

{context}
""",
    )

### 6.1 Integrando schema linking + contexto focado + geração

In [107]:
class SQLGenerationContext(BaseModel):
    """Resultado completo da etapa de geração de SQL."""

    schema_link: SchemaLink = Field(
        description="Resultado do schema linking utilizado na geração."
    )

    focused_context: str = Field(
        description="Contexto focado enviado ao agente gerador de SQL."
    )

    generation: SQLGeneration = Field(
        description="Consulta SQL gerada e seu reasoning."
    )

In [134]:
async def generate_sql(
    question: str,
    db_path: str = DB_PATH,
) -> SQLGenerationContext:
    """
    Executa o fluxo de schema linking, construção do contexto focado
    e geração da consulta SQL.

    Args:
        question: Pergunta do usuário em linguagem natural.
        db_path: Caminho para o banco SQLite.

    Returns:
        Estrutura contendo o schema linking, o contexto focado
        e a consulta SQL gerada.

    Raises:
        RuntimeError: Caso o modelo retorne uma resposta inválida.
    """
    try:
        link_result = await run_agent_with_retry(
            schema_link_agent,
            question,
        )
    
        linked = link_result.output

        context = build_focused_context(
            linked,
            db_path,
        )

        sql_agent = build_sql_generation_agent(context)

        sql_result = await run_agent_with_retry(
            sql_agent,
            question,
        )

        return SQLGenerationContext(
            schema_link=linked,
            focused_context=context,
            generation=sql_result.output,
        )

    except UnexpectedModelBehavior as exc:
        raise RuntimeError(
            "O modelo retornou uma resposta inválida durante a geração do SQL."
        ) from exc

### 6.2 Testando geração de consulta SQL

In [89]:
question = "Top 10 filmes com maior receita em R$"

generated = await generate_sql(question)

print("SQL:")
print(generated.sql)

print("\nReasoning:")
print(generated.reasoning)

SQL:
WITH cleaned AS (
    SELECT dm.titulo, fmp.receita_brl
    FROM dim_movies dm
    JOIN fact_movies_performance fmp ON dm.sk_movie_id = fmp.sk_movie_id
    WHERE fmp.receita_brl IS NOT NULL
)
SELECT titulo, receita_brl
FROM cleaned
ORDER BY receita_brl DESC
LIMIT 10;

Reasoning:
Juntei dim_movies e fact_movies_performance pela chave sk_movie_id, removi linhas com receita nula, ordenei pela receita em ordem decrescente e limitei a 10 registros para obter os 10 filmes com maior receita em reais.


In [90]:
question = "Qual dupla ator-diretor mais trabalhou junta?"

generated = await generate_sql(question)

print("SQL:")
print(generated.sql)

print("\nReasoning:")
print(generated.reasoning)

SQL:
WITH actor_movies AS (
  SELECT bmp.sk_movie_id, p.sk_person_id AS actor_id, p.nome_pessoa AS actor_name
  FROM bridge_movie_person bmp
  JOIN dim_people p ON bmp.sk_person_id = p.sk_person_id
  WHERE p.tipo_pessoa = 'Ator'
),
 director_movies AS (
  SELECT bmp.sk_movie_id, p.sk_person_id AS director_id, p.nome_pessoa AS director_name
  FROM bridge_movie_person bmp
  JOIN dim_people p ON bmp.sk_person_id = p.sk_person_id
  WHERE p.tipo_pessoa = 'Diretor'
)
SELECT a.actor_name, d.director_name, COUNT(DISTINCT a.sk_movie_id) AS colaboracoes
FROM actor_movies a
JOIN director_movies d ON a.sk_movie_id = d.sk_movie_id
GROUP BY a.actor_id, d.director_id
ORDER BY colaboracoes DESC
LIMIT 1;

Reasoning:
A consulta utiliza duas CTEs para isolar os filmes em que cada ator e cada diretor participaram, filtrando pela coluna tipo_pessoa da dim_people. Em seguida, faz o join entre as duas CTEs pelo sk_movie_id para obter todas as combinações ator-diretor por filme. Agrupa por par ator-diretor, c

## 7. Validação e execução segura do SQL

Antes de executar qualquer consulta gerada pelo modelo, o SQL é validado para permitir apenas operações de leitura e bloquear comandos potencialmente destrutivos.

In [91]:
FORBIDDEN_SQL_KEYWORDS = {
    "INSERT",
    "UPDATE",
    "DELETE",
    "DROP",
    "ALTER",
    "CREATE",
    "REPLACE",
    "TRUNCATE",
    "ATTACH",
    "DETACH",
    "VACUUM",
}


def validate_sql(sql: str) -> None:
    """
    Valida se uma consulta SQL pode ser executada com segurança.

    A função permite apenas consultas de leitura iniciadas por SELECT ou WITH
    e bloqueia comandos que possam modificar a estrutura ou os dados do banco.

    Args:
        sql: Consulta SQL que será validada.

    Raises:
        ValueError: Caso a consulta esteja vazia, contenha múltiplas instruções,
            não seja uma consulta de leitura ou possua comandos proibidos.
    """
    if not sql or not sql.strip():
        raise ValueError("A consulta SQL está vazia.")

    normalized_sql = sql.strip()

    # Remove apenas o ponto e vírgula final, caso exista
    sql_without_final_semicolon = normalized_sql.rstrip(";").strip()

    # Impede múltiplas instruções SQL
    if ";" in sql_without_final_semicolon:
        raise ValueError(
            "A consulta contém múltiplas instruções SQL."
        )

    # Remove comentários para evitar que palavras-chave em comentários
    # interfiram na validação
    sql_without_comments = re.sub(
        r"--.*?$|/\*.*?\*/",
        " ",
        sql_without_final_semicolon,
        flags=re.MULTILINE | re.DOTALL,
    )

    upper_sql = sql_without_comments.upper().strip()

    if not (
        upper_sql.startswith("SELECT")
        or upper_sql.startswith("WITH")
    ):
        raise ValueError(
            "Apenas consultas de leitura iniciadas por SELECT ou WITH são permitidas."
        )

    for keyword in FORBIDDEN_SQL_KEYWORDS:
        if re.search(rf"\b{keyword}\b", upper_sql):
            raise ValueError(
                f"Comando SQL proibido detectado: {keyword}."
            )

### 7.1 Execução segura da consulta

As consultas validadas são executadas em uma conexão SQLite aberta em modo somente leitura, evitando alterações acidentais no banco.

In [94]:
def execute_sql(
    db_path: str,
    sql: str,
) -> list[dict[str, object]]:
    """
    Executa uma consulta SQL de leitura em um banco SQLite.

    Antes da execução, a consulta é validada e o banco é aberto em modo
    somente leitura.

    Args:
        db_path: Caminho para o banco SQLite.
        sql: Consulta SQL que será executada.

    Returns:
        Lista de dicionários, em que cada dicionário representa uma linha
        retornada pela consulta.

    Raises:
        ValueError: Caso a consulta SQL não passe pela validação de segurança.
        sqlite3.Error: Caso ocorra um erro durante a execução da consulta.
    """
    validate_sql(sql)

    database_uri = Path(db_path).resolve().as_uri() + "?mode=ro"

    conn = sqlite3.connect(
        database_uri,
        uri=True,
    )

    conn.row_factory = sqlite3.Row

    try:
        cursor = conn.execute(sql)

        return [
            dict(row)
            for row in cursor.fetchall()
        ]

    finally:
        conn.close()

### 7.2 Testando validação do SQL

In [95]:
safe_queries = [
    "SELECT * FROM dim_movies LIMIT 10;",
    """
    WITH top_movies AS (
        SELECT sk_movie_id, receita_brl
        FROM fact_movies_performance
    )
    SELECT *
    FROM top_movies
    ORDER BY receita_brl DESC
    LIMIT 10;
    """,
]

for query in safe_queries:
    validate_sql(query)
    print("Consulta válida.")

Consulta válida.
Consulta válida.


In [96]:
unsafe_queries = [
    "DELETE FROM dim_movies;",
    "DROP TABLE dim_movies;",
    "UPDATE dim_movies SET titulo = 'Teste';",
    "SELECT * FROM dim_movies; DROP TABLE dim_movies;",
]

for query in unsafe_queries:
    try:
        validate_sql(query)
    except ValueError as exc:
        print(f"Bloqueada: {exc}")

Bloqueada: Apenas consultas de leitura iniciadas por SELECT ou WITH são permitidas.
Bloqueada: Apenas consultas de leitura iniciadas por SELECT ou WITH são permitidas.
Bloqueada: Apenas consultas de leitura iniciadas por SELECT ou WITH são permitidas.
Bloqueada: A consulta contém múltiplas instruções SQL.


### 7.3 Testando execução segura

In [97]:
sql = """
SELECT titulo, ano_lancamento
FROM dim_movies
LIMIT 5;
"""

result = execute_sql(
    DB_PATH,
    sql,
)

result

[{'titulo': 'Rings', 'ano_lancamento': 2017},
 {'titulo': 'Mixtape', 'ano_lancamento': 2021},
 {'titulo': 'Grizzly Ii: Revenge', 'ano_lancamento': 2020},
 {'titulo': 'Billy Joel - Live At Yankee Stadium', 'ano_lancamento': 2022},
 {'titulo': 'Bad Boys For Life', 'ano_lancamento': 2020}]

In [98]:
sql = """
DELETE FROM dim_movies;
"""

try:
    execute_sql(
        DB_PATH,
        sql,
    )

except Exception as exc:
    print(type(exc).__name__)
    print(exc)

ValueError
Apenas consultas de leitura iniciadas por SELECT ou WITH são permitidas.


In [99]:
database_uri = Path(DB_PATH).resolve().as_uri() + "?mode=ro"

conn = sqlite3.connect(
    database_uri,
    uri=True,
)

try:
    conn.execute(
        "DELETE FROM dim_movies WHERE 1 = 0;"
    )

except sqlite3.Error as exc:
    print(exc)

finally:
    conn.close()

attempt to write a readonly database


## 8. Verificação e correção da consulta

Após a execução, o resultado é verificado para identificar erros de SQL ou inconsistências entre a pergunta, a consulta gerada e os dados retornados.

Quando necessário, uma nova consulta pode ser gerada utilizando o erro ou a justificativa da verificação como contexto.

In [ ]:
# Modelo de verificação
class SQLVerification(BaseModel):
    """Resultado da verificação de uma consulta SQL executada."""

    valid: bool = Field(
        description=(
            "Indica se a consulta e o resultado são adequados "
            "para responder à pergunta."
        )
    )

    reason: str = Field(
        description=(
            "Explicação breve sobre o motivo da consulta ser considerada "
            "válida ou precisar de correção."
        )
    )
    
    verified_by_llm: bool = True

### 8.1 Verificação do resultado

O verificador recebe a pergunta original, o SQL executado e uma pequena amostra do resultado.

Ele não pode modificar dados nem inventar informações; sua função é apenas avaliar se a consulta produz informações compatíveis com a pergunta.

In [148]:
def basic_result_check(
    sql: str,
    rows: list[dict[str, object]],
) -> None:
    """
    Realiza verificações básicas sobre uma consulta já executada.

    Args:
        sql: Consulta SQL executada.
        rows: Resultado retornado pelo banco.

    Raises:
        ValueError: Caso a estrutura básica do resultado seja inválida.
    """
    validate_sql(sql)

    if not isinstance(rows, list):
        raise ValueError(
            "O resultado da consulta deve ser uma lista."
        )

    if rows and not all(
        isinstance(row, dict)
        for row in rows
    ):
        raise ValueError(
            "As linhas retornadas devem possuir estrutura de dicionário."
        )

In [101]:
# Limita a quantidade de linhas enviadas ao modelo
def get_result_sample(
    rows: list[dict[str, object]],
    limit: int = 5,
) -> list[dict[str, object]]:
    """
    Retorna uma pequena amostra do resultado de uma consulta.

    Args:
        rows: Linhas retornadas pela consulta SQL.
        limit: Quantidade máxima de linhas utilizadas na amostra.

    Returns:
        Subconjunto das linhas retornadas.
    """
    if limit <= 0:
        return []

    return rows[:limit]

In [144]:
def build_sql_verification_agent() -> Agent:
    """
    Cria o agente responsável por verificar se uma consulta SQL
    e seu resultado são adequados para responder à pergunta.

    Returns:
        Agente configurado para verificação estruturada.
    """
    return Agent(
        pydantic_model,
        output_type=PromptedOutput(SQLVerification),
        instructions="""
Você é responsável por verificar resultados de um sistema Text-to-SQL.

Sua tarefa NÃO é responder à pergunta do usuário.

Analise:
- a pergunta original;
- a consulta SQL executada;
- a quantidade de registros retornados;
- uma amostra real do resultado.

Regras:

1. Não invente dados.
2. Não altere nem gere uma nova consulta SQL.
3. Verifique se o SQL está semanticamente relacionado à pergunta.
4. Verifique se as colunas retornadas são suficientes para responder à pergunta.
5. Verifique se agregações, ordenações e limites são coerentes quando a
   pergunta exigir essas operações.
6. Um resultado vazio não significa automaticamente que a consulta está errada.
7. Considere somente as informações fornecidas.
8. Seja breve na justificativa.
9. Não avalie se nomes, valores ou resultados parecem plausíveis com base em
   conhecimento externo. Verifique apenas se o SQL e os dados retornados
   são estruturalmente suficientes para responder à pergunta.
10. Nunca use conhecimento próprio para confirmar ou rejeitar um resultado.
    Os dados retornados pelo banco são a única fonte factual disponível.
11. Considere a consulta válida quando sua estrutura e os dados retornados forem
    suficientes para responder à pergunta, sem exigir justificativas extensas.
""",
    )

In [147]:
async def verify_sql_result(
    question: str,
    sql: str,
    rows: list[dict[str, object]],
) -> SQLVerification:
    """
    Verifica semanticamente uma consulta SQL e seu resultado.

    Caso o modelo de verificação esteja indisponível, aplica uma
    verificação determinística básica para permitir a continuidade
    segura do pipeline.

    Args:
        question: Pergunta original do usuário.
        sql: Consulta SQL executada.
        rows: Linhas retornadas pelo banco.

    Returns:
        Resultado estruturado da verificação.
    """
    sample = get_result_sample(rows)

    verification_input = f"""
PERGUNTA:
{question}

SQL EXECUTADO:
{sql}

QUANTIDADE DE REGISTROS:
{len(rows)}

AMOSTRA DO RESULTADO:
{sample}
"""

    agent = build_sql_verification_agent()

    try:
        result = await run_agent_with_retry(
            agent,
            verification_input,
        )

        return result.output

    except UnexpectedModelBehavior:
        return SQLVerification(
            valid=True,
            reason=(
                "A verificação semântica pelo modelo não pôde ser concluída. "
                "A consulta passou pela validação de segurança e foi executada "
                "com sucesso no banco SQLite."
            ),
            verified_by_llm=False,
        )

### 8.2 Testando a Verificação das consultas

In [109]:
question = "Top 10 filmes com maior receita em R$"

generated = await generate_sql(question)

rows = execute_sql(
    DB_PATH,
    generated.generation.sql,
)

verification = await verify_sql_result(
    question,
    generated.generation.sql,
    rows,
)

print("Válido:", verification.valid)
print("Motivo:", verification.reason)

Válido: True
Motivo: A consulta recupera corretamente os 10 filmes com maior receita em reais (BRL), realizando o join adequado, filtrando valores nulos, ordenando de forma decrescente e aplicando o limite de 10 registros. As colunas retornadas (titulo e receita_brl) são suficientes para responder à pergunta.


### 8.3 Correção da consulta

Quando ocorre um erro de execução ou a verificação identifica uma inconsistência, um agente recebe a consulta original, o problema encontrado e o contexto focado para gerar uma versão corrigida.

In [120]:
def build_sql_correction_agent(
    context: str,
) -> Agent:
    """
    Cria um agente responsável por corrigir uma consulta SQL.

    Args:
        context: Contexto focado utilizado originalmente para gerar a consulta.

    Returns:
        Agente configurado para gerar uma consulta corrigida.
    """
    return Agent(
        pydantic_model,
        output_type=PromptedOutput(SQLGeneration),
        instructions=f"""
Você é responsável por corrigir consultas SQL em um sistema Text-to-SQL.

Você receberá:
- a pergunta original;
- uma consulta SQL que apresentou problema;
- a descrição do problema;
- o contexto disponível do banco.

Sua tarefa é gerar uma nova consulta SQL corrigida.

Regras:

1. Gere apenas consultas de leitura.
2. A consulta deve começar com SELECT ou WITH.
3. Nunca utilize comandos que modifiquem o banco.
4. Nunca invente tabelas ou colunas.
5. Utilize somente tabelas e colunas presentes no contexto.
6. Utilize somente valores categóricos observados no contexto.
7. Considere explicitamente o problema informado ao corrigir a consulta.
8. Não repita o mesmo erro da consulta anterior.
9. Utilize sintaxe compatível com SQLite.
10. Utilize aliases quando a mesma tabela desempenhar papéis diferentes.
11. Não responda à pergunta em linguagem natural.
12. Retorne somente a consulta corrigida e uma justificativa breve.
13. Não invente valores de filtro que não estejam presentes na pergunta
    ou observados no contexto.
14. Trate valores NULL quando isso for necessário para corrigir a consulta.

CONTEXTO DISPONÍVEL:

{context}
""",
    )

In [121]:
async def correct_sql(
    question: str,
    original_sql: str,
    problem: str,
    context: str,
) -> SQLGeneration:
    """
    Gera uma nova consulta SQL com base em um problema identificado.

    Args:
        question: Pergunta original do usuário.
        original_sql: Consulta SQL que precisa ser corrigida.
        problem: Erro ou inconsistência encontrada.
        context: Contexto focado disponível para geração do SQL.

    Returns:
        Nova consulta SQL estruturada.
    """
    correction_input = f"""
PERGUNTA:
{question}

SQL ORIGINAL:
{original_sql}

PROBLEMA IDENTIFICADO:
{problem}
"""

    agent = build_sql_correction_agent(context)

    result = await run_agent_with_retry(
        agent,
        correction_input,
    )

    return result.output

## 9. Pipeline Text-to-SQL

Esta seção integra as etapas de schema linking, geração, validação, execução, verificação e correção em um único fluxo.

Para evitar chamadas excessivas ao modelo, escolhi permitir no máximo uma tentativa de correção por pergunta.

## 9.1 Modelo da resposta final

A resposta final é produzida exclusivamente a partir dos dados retornados pela consulta SQL validada e executada.

O modelo não deve adicionar informações externas, inferir dados ausentes ou complementar a resposta com conhecimento próprio.

In [159]:
# Modelo de resposta final
class FinalAnswer(BaseModel):
    """Resposta final baseada exclusivamente no resultado da consulta."""

    answer: str = Field(
        description=(
            "Resposta em linguagem natural baseada exclusivamente "
            "nos dados retornados pela consulta SQL."
        )
    )

### 9.2 Geração da resposta em NL

In [154]:
def build_fallback_answer(
    rows: list[dict[str, object]],
) -> str:
    """
    Constrói uma resposta textual diretamente a partir dos registros
    retornados pelo banco, sem utilizar um modelo de linguagem.

    Args:
        rows: Registros retornados pela consulta SQL.

    Returns:
        Representação textual dos dados.
    """
    if not rows:
        return "Nenhum registro correspondente foi encontrado."

    lines = []

    for index, row in enumerate(rows, start=1):
        values = [
            f"{column}: {value}"
            for column, value in row.items()
        ]

        lines.append(
            f"{index}. " + " | ".join(values)
        )

    return "\n".join(lines)

In [156]:
async def generate_final_answer(
    question: str,
    sql: str,
    rows: list[dict[str, object]],
) -> FinalAnswer:
    """
    Gera uma resposta em linguagem natural baseada exclusivamente
    nos registros retornados pelo banco.

    Caso o provider falhe, utiliza uma representação determinística
    dos dados como fallback.

    Args:
        question: Pergunta original do usuário.
        sql: Consulta SQL efetivamente executada.
        rows: Registros reais retornados pelo banco.

    Returns:
        Resposta final estruturada.
    """
    final_input = f"""
PERGUNTA:
{question}

SQL EXECUTADO:
{sql}

DADOS RETORNADOS PELO BANCO:
{rows}
"""

    agent = build_final_answer_agent()

    try:
        result = await run_agent_with_retry(
            agent,
            final_input,
        )

        return result.output

    except UnexpectedModelBehavior:
        return FinalAnswer(
            answer=build_fallback_answer(rows)
        )

### 9.3 Modelo do resultado do pipeline

In [152]:
# Cria um modelo para o resultado final do pipelina
class SQLVerification(BaseModel):
    """Resultado da verificação de uma consulta SQL."""

    valid: bool = Field(
        description="Indica se a consulta parece adequada para responder à pergunta."
    )

    reason: str = Field(
        description="Justificativa breve da verificação."
    )

    verified_by_llm: bool = Field(
        default=True,
        description="Indica se a verificação foi realizada pelo modelo."
    )

In [137]:
async def run_agent_with_retry(
    agent: Agent,
    prompt: str,
    max_retries: int = 1,
    delay: float = 2.0,
):
    """
    Executa um agente com uma quantidade limitada de novas tentativas
    quando o provider retorna uma resposta inválida.

    Args:
        agent: Agente PydanticAI que será executado.
        prompt: Entrada enviada ao agente.
        max_retries: Número máximo de novas tentativas.
        delay: Intervalo inicial entre as tentativas, em segundos.

    Returns:
        Resultado retornado pelo agente.

    Raises:
        UnexpectedModelBehavior: Caso todas as tentativas falhem.
    """
    for attempt in range(max_retries + 1):
        try:
            return await agent.run(prompt)

        except UnexpectedModelBehavior:
            if attempt == max_retries:
                raise

            await asyncio.sleep(delay * (attempt + 1))

In [149]:
class TextToSQLResult(BaseModel):
    """Resultado completo do pipeline Text-to-SQL."""

    question: str
    sql: str
    rows: list[dict[str, object]]
    verification: SQLVerification
    corrected: bool
    answer: FinalAnswer

In [ ]:
async def run_text_to_sql(
    question: str,
    db_path: str = DB_PATH,
) -> TextToSQLResult:
    """
    Executa o pipeline completo de Text-to-SQL.

    O fluxo realiza:
    1. schema linking;
    2. construção do contexto focado;
    3. geração do SQL;
    4. validação e execução;
    5. verificação do resultado;
    6. uma tentativa de correção, quando necessária;
    7. geração da resposta final.

    Args:
        question: Pergunta do usuário em linguagem natural.
        db_path: Caminho para o banco SQLite.

    Returns:
        Resultado final do pipeline Text-to-SQL.

    Raises:
        RuntimeError: Caso o pipeline não consiga produzir uma
            consulta executável ou o provider falhe.
    """
    try:
        generated = await generate_sql(
            question=question,
            db_path=db_path,
        )

    except RuntimeError as exc:
        raise RuntimeError(
            f"Não foi possível iniciar o pipeline: {exc}"
        ) from exc

    original_sql = generated.generation.sql

    # Primeira tentativa de execução
    try:
        rows = execute_sql(
            db_path,
            original_sql,
        )

    except (ValueError, sqlite3.Error) as exc:
        corrected = await correct_sql(
            question=question,
            original_sql=original_sql,
            problem=str(exc),
            context=generated.focused_context,
        )

        try:
            corrected_rows = execute_sql(
                db_path,
                corrected.sql,
            )

        except (ValueError, sqlite3.Error) as correction_exc:
            raise RuntimeError(
                "A consulta corrigida também apresentou erro durante a execução."
            ) from correction_exc

        corrected_verification = await verify_sql_result(
            question=question,
            sql=corrected.sql,
            rows=corrected_rows,
        )

        answer = await generate_final_answer(
            question=question,
            sql=corrected.sql,
            rows=corrected_rows,
        )

        return TextToSQLResult(
            question=question,
            sql=corrected.sql,
            rows=corrected_rows,
            verification=corrected_verification,
            corrected=True,
            answer=answer,
        )

    verification = await verify_sql_result(
        question=question,
        sql=original_sql,
        rows=rows,
    )

    # Consulta original considerada correta
    if verification.valid:
        answer = await generate_final_answer(
            question=question,
            sql=original_sql,
            rows=rows,
        )

        return TextToSQLResult(
            question=question,
            sql=original_sql,
            rows=rows,
            verification=verification,
            corrected=False,
            answer=answer,
        )

    # SQL executou, mas foi considerado semanticamente inadequado
    corrected = await correct_sql(
        question=question,
        original_sql=original_sql,
        problem=verification.reason,
        context=generated.focused_context,
    )

    try:
        corrected_rows = execute_sql(
            db_path,
            corrected.sql,
        )

    except (ValueError, sqlite3.Error) as exc:
        raise RuntimeError(
            "A consulta corrigida apresentou erro durante a execução."
        ) from exc

    corrected_verification = await verify_sql_result(
        question=question,
        sql=corrected.sql,
        rows=corrected_rows,
    )
    
    if not corrected_verification.valid:
        raise RuntimeError(
            "A consulta corrigida ainda foi considerada semanticamente inadequada."
        )

    answer = await generate_final_answer(
        question=question,
        sql=corrected.sql,
        rows=corrected_rows,
    )

    return TextToSQLResult(
        question=question,
        sql=corrected.sql,
        rows=corrected_rows,
        verification=corrected_verification,
        corrected=True,
        answer=answer,
    )

### 9.4 Testando pipeline

In [150]:
question = "Top 10 filmes com maior receita em R$"

result = await run_text_to_sql(question)

print("RESPOSTA FINAL:")
print(result.answer.answer)

print("\nSQL:")
print(result.sql)

print("\nRESULTADO:")
print(result.rows)

print("\nVERIFICAÇÃO:")
print(result.verification)

print("\nCORRIGIDO:")
print(result.corrected)

RESPOSTA FINAL:
Os 10 filmes com maior receita em R$ são:
1. Avatar: The Way Of Water – R$ 12.390.136.500,54
2. Avengers: Endgame – R$ 11.094.720.000,00
3. Spider-man: No Way Home – R$ 10.977.782.882,74
4. Avengers: Infinity War – R$ 7.190.430.847,63
5. Top Gun: Maverick – R$ 7.160.804.869,01
6. Barbie – R$ 6.856.159.007,38
7. The Super Mario Bros. Movie – R$ 6.838.413.799,10
8. The Lion King – R$ 6.227.552.146,58
9. Frozen Ii – R$ 6.094.028.191,32
10. Jurassic World: Fallen Kingdom – R$ 4.934.822.930,85

SQL:
SELECT dm.titulo AS filme,
       fmp.receita_brl AS receita_brl
FROM dim_movies dm
JOIN fact_movies_performance fmp ON dm.sk_movie_id = fmp.sk_movie_id
WHERE fmp.receita_brl IS NOT NULL
ORDER BY fmp.receita_brl DESC
LIMIT 10;

RESULTADO:
[{'filme': 'Avatar: The Way Of Water', 'receita_brl': 12390136500.54}, {'filme': 'Avengers: Endgame', 'receita_brl': 11094720000}, {'filme': 'Spider-man: No Way Home', 'receita_brl': 10977782882.74}, {'filme': 'Avengers: Infinity War', 'receita_

In [157]:
question = "Qual dupla ator-diretor mais trabalhou junta?"

result = await run_text_to_sql(question)

print("RESPOSTA FINAL:")
print(result.answer.answer)

print("\nSQL:")
print(result.sql)

print("\nRESULTADO:")
print(result.rows)

print("\nVERIFICAÇÃO:")
print(result.verification)

print("\nCORRIGIDO:")
print(result.corrected)

ValidationError: 1 validation error for TextToSQLResult
verification
  Input should be a valid dictionary or instance of SQLVerification [type=model_type, input_value=SQLVerification(valid=Tru..., verified_by_llm=False), input_type=SQLVerification]
    For further information visit https://errors.pydantic.dev/2.13/v/model_type